# RFD Urban Digital Twin Analysis

This notebook builds compact exam pipeline for RFD-based consistency profiling on urban air-quality data.

Scope:
- simplified conceptual Urban Digital Twin;
- Beijing air-quality subset with two stations;
- interpretable approximate rules, not causal claims;
- lightweight discovery over reduced deterministic sample.


## Dataset Configuration

Selected subset:
- stations: `Aotizhongxin`, `Changping`
- period: `2013-03-01` to `2017-02-28`
- preprocessing and profiling use the complete common station coverage
- RFD experiment sample: 1500 rows balanced across stations

Quadratic RFD experiments use the unchanged deterministic balanced 1500-row sample.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [current, *current.parents] if (path / 'AGENTS.md').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

for path in [DATA_PROCESSED, RESULTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [2]:
from src.experiments import (
    prepare_rfd_sample,
    run_candidate_validation,
    run_lightweight_discovery,
    run_station_comparison,
    run_threshold_comparison,
    select_top_rule_violations,
    export_violation_examples,
    run_candidate_metrics,
    run_bootstrap_validation,
    run_train_test_validation,
    run_window_evolution,
    run_violation_analysis,
)
from src.preprocessing import prepare_udt_dataset
from src.profiling import (
    correlation_matrix,
    data_types_summary,
    dataset_shape_summary,
    hour_distribution,
    missing_values_summary,
    numeric_summary,
    station_distribution,
    time_slot_distribution,
    unique_values_summary,
)
from src.visualization import (
    plot_correlation_matrix,
    plot_missing_values,
    plot_pm25_timeseries,
    plot_pollutant_distribution,
)


## Step 1-2: Preprocessing

In [3]:
cleaned_df, pre_drop_missing = prepare_udt_dataset(
    raw_dir=DATA_RAW,
    processed_path=DATA_PROCESSED / 'udt_rfd_dataset.csv',
)
sample_df = prepare_rfd_sample(cleaned_df)
sample_df.to_csv(DATA_PROCESSED / 'udt_rfd_sample.csv', index=False)

print('Cleaned rows:', len(cleaned_df))
print('Cleaned columns:', cleaned_df.shape[1])
print('RFD sample rows:', len(sample_df))
display(pre_drop_missing)


Cleaned rows: 66619
Cleaned columns: 11
RFD sample rows: 1500


,column,missing_count,missing_rate
0,O3,2323,0.033125
1,PM2.5,1699,0.024227
2,NO2,1690,0.024099
3,PM10,1300,0.018538
4,DEWP,73,0.001041
5,TEMP,73,0.001041
6,WSPM,57,0.000813
7,datetime,0,0.000000
8,hour,0,0.000000
9,station,0,0.000000


## Step 3: Profiling

In [4]:
shape_df = dataset_shape_summary(cleaned_df)
dtypes_df = data_types_summary(cleaned_df)
missing_df = missing_values_summary(cleaned_df)
unique_df = unique_values_summary(cleaned_df)
numeric_df = numeric_summary(cleaned_df)
station_df = station_distribution(cleaned_df)
hour_df = hour_distribution(cleaned_df)
time_slot_df = time_slot_distribution(cleaned_df)
corr_df = correlation_matrix(
    cleaned_df,
    numeric_columns=['hour', 'PM2.5', 'PM10', 'NO2', 'O3', 'TEMP', 'DEWP', 'WSPM'],
)

shape_df.to_csv(RESULTS_DIR / 'profile_shape_summary.csv', index=False)
dtypes_df.to_csv(RESULTS_DIR / 'profile_dtypes_summary.csv', index=False)
missing_df.to_csv(RESULTS_DIR / 'profile_missing_summary.csv', index=False)
unique_df.to_csv(RESULTS_DIR / 'profile_unique_values_summary.csv', index=False)
numeric_df.to_csv(RESULTS_DIR / 'profile_numeric_summary.csv', index=False)
station_df.to_csv(RESULTS_DIR / 'profile_station_distribution.csv', index=False)
hour_df.to_csv(RESULTS_DIR / 'profile_hour_distribution.csv', index=False)
time_slot_df.to_csv(RESULTS_DIR / 'profile_time_slot_distribution.csv', index=False)
corr_df.to_csv(RESULTS_DIR / 'profile_correlation_matrix.csv')

plot_missing_values(missing_df, FIGURES_DIR / 'missing_values.png')
plot_pollutant_distribution(cleaned_df, 'PM2.5', FIGURES_DIR / 'pm25_distribution.png')
plot_pollutant_distribution(cleaned_df, 'NO2', FIGURES_DIR / 'no2_distribution.png')
plot_pm25_timeseries(cleaned_df, FIGURES_DIR / 'pm25_timeseries.png')
plot_correlation_matrix(corr_df, FIGURES_DIR / 'correlation_matrix.png')

print('Dataset shape')
display(shape_df)
print('Data types')
display(dtypes_df)
print('Missing values after cleaning')
display(missing_df)
print('Unique values')
display(unique_df)
print('Numeric summary')
display(numeric_df)
print('Station distribution')
display(station_df)
print('Hour distribution')
display(hour_df)
print('Time-slot distribution')
display(time_slot_df)


Dataset shape


,metric,value
0,rows,66619
1,columns,11


Data types


,column,dtype
0,DEWP,float64
1,NO2,float64
2,O3,float64
3,PM10,float64
4,PM2.5,float64
5,TEMP,float64
6,WSPM,float64
7,datetime,datetime64[us]
8,hour,int64
9,station,str


Missing values after cleaning


...

,station,rows
0,Aotizhongxin,32779
1,Changping,33840


Hour distribution


,hour,rows
0,0,2813
1,1,2811
2,2,2740
3,3,2722
4,4,2435
5,5,2749
6,6,2809
7,7,2801
8,8,2810
9,9,2828


Time-slot distribution


,time_slot,rows
0,night,16270
1,morning,16823
2,afternoon,16645
3,evening,16881


## Step 4-6: Candidate RFD Validation

Threshold sets are defined in `src/rfd.py` as `strict`, `medium`, and `relaxed`.
Candidate rules are evaluated on balanced 1500-row sample to keep pairwise validation tractable.


In [5]:
candidate_df, candidate_violations = run_candidate_validation(
    sample_df,
    RESULTS_DIR / 'rfd_candidate_results.csv',
)

candidate_table = candidate_df.loc[:, ['rule_label', 'support', 'confidence', 'violations', 'interpretation']].copy()
for column in ['support', 'confidence']:
    candidate_table[column] = candidate_table[column].round(4)

display(candidate_table)


,rule_label,support,confidence,violations,interpretation
2,"station, PM2.5, NO2 -> PM10",0.0251,0.4834,14598,"Within one station, similar PM2.5 and NO2 shou..."
1,"station, time_slot, PM2.5 -> PM10",0.0185,0.4488,11438,"Within one station and time slot, similar PM2...."
0,"station, PM2.5 -> PM10",0.0733,0.4336,46681,"Within one station, particulate measures shoul..."
3,"station, hour, TEMP, WSPM -> O3",0.0043,0.2755,3524,"Within one station and similar hour, temperatu..."
4,"station, time_slot, TEMP, WSPM -> O3",0.0084,0.2671,6905,"Within one station and time slot, similar temp..."
5,"station, time_slot, NO2 -> O3",0.0273,0.2320,23600,"Within one station and time slot, similar NO2 ..."
6,"station, TEMP, DEWP, WSPM -> PM2.5",0.0066,0.1517,6269,"Within one station, similar meteorological con..."


In [6]:
candidate_metrics_df = run_candidate_metrics(
    sample_df,
    RESULTS_DIR / 'rfd_candidate_metrics.csv',
    FIGURES_DIR / 'rfd_lift_vs_baseline.png',
)

metrics_table = candidate_metrics_df.loc[:, [
    'representation', 'rule_label', 'support', 'confidence',
    'baseline_confidence', 'baseline_confidence_std', 'lift', 'violation_rate'
]].copy()
for column in ['support', 'confidence', 'baseline_confidence', 'baseline_confidence_std', 'lift', 'violation_rate']:
    metrics_table[column] = metrics_table[column].round(4)

display(metrics_table)


,representation,rule_label,support,confidence,baseline_confidence,baseline_confidence_std,lift,violation_rate
0,raw,"station, PM2.5, NO2 -> PM10",0.0251,0.4834,0.1493,0.0048,3.2371,0.5166
1,raw,"station, time_slot, PM2.5 -> PM10",0.0185,0.4488,0.1477,0.0038,3.0377,0.5512
2,raw,"station, PM2.5 -> PM10",0.0733,0.4336,0.1475,0.0029,2.9404,0.5664
3,raw,"station, hour, TEMP, WSPM -> O3",0.0043,0.2755,0.1618,0.0058,1.7022,0.7245
4,raw,"station, time_slot, TEMP, WSPM -> O3",0.0084,0.2671,0.1629,0.0050,1.6394,0.7329
5,raw,"station, time_slot, NO2 -> O3",0.0273,0.2320,0.1615,0.0029,1.4362,0.7680
6,raw,"station, TEMP, DEWP, WSPM -> PM2.5",0.0066,0.1517,0.1459,0.0053,1.0399,0.8483
7,binned,"station, PM2.5_bin, NO2_bin -> PM10_bin",0.0786,0.6886,0.3331,0.0017,2.0672,0.3114
8,binned,"station, time_slot, PM2.5_bin -> PM10_bin",0.0417,0.6376,0.3327,0.0023,1.9167,0.3624
9,binned,"station, PM2.5_bin -> PM10_bin",0.1671,0.6294,0.3325,0.0006,1.8927,0.3706


## Step 7: Lightweight Discovery

In [7]:
discovery_df = run_lightweight_discovery(
    sample_df,
    RESULTS_DIR / 'rfd_discovered_top10.csv',
)

if discovery_df.empty:
    print('No discovered RFD met support >= 0.01 and confidence >= 0.85 under medium thresholds.')
else:
    display(discovery_df.round(4))


No discovered RFD met support >= 0.01 and confidence >= 0.85 under medium thresholds.


## Step 8-9: Threshold and Station Comparisons

In [8]:
threshold_df = run_threshold_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_threshold_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_threshold.png',
)
station_df_rfd = run_station_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_station_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_station.png',
)

display(threshold_df[['rule_label', 'threshold_set', 'support', 'confidence', 'violations']].round(4))
display(station_df_rfd[['rule_label', 'station_scope', 'support', 'confidence', 'violations']].round(4))


,rule_label,threshold_set,support,confidence,violations
0,"station, PM2.5 -> PM10",medium,0.0733,0.4336,46681
1,"station, PM2.5 -> PM10",relaxed,0.1018,0.5758,48525
2,"station, PM2.5 -> PM10",strict,0.0409,0.3384,30447
3,"station, PM2.5, NO2 -> PM10",medium,0.0251,0.4834,14598
4,"station, PM2.5, NO2 -> PM10",relaxed,0.0465,0.6295,19351
5,"station, PM2.5, NO2 -> PM10",strict,0.0080,0.3837,5548
6,"station, TEMP, DEWP, WSPM -> PM2.5",medium,0.0066,0.1517,6269
7,"station, TEMP, DEWP, WSPM -> PM2.5",relaxed,0.0179,0.2162,15735
8,"station, TEMP, DEWP, WSPM -> PM2.5",strict,0.0011,0.0908,1151
9,"station, hour, TEMP, WSPM -> O3",medium,0.0043,0.2755,3524


,rule_label,station_scope,support,confidence,violations
0,PM2.5 -> PM10,Aotizhongxin,0.1340,0.4138,22069
1,PM2.5 -> PM10,Changping,0.1594,0.4503,24612
2,"PM2.5, NO2 -> PM10",Aotizhongxin,0.0380,0.4798,5552
3,"PM2.5, NO2 -> PM10",Changping,0.0626,0.4856,9046
4,"TEMP, DEWP, WSPM -> PM2.5",Aotizhongxin,0.0134,0.1396,3243
5,"TEMP, DEWP, WSPM -> PM2.5",Changping,0.0129,0.1643,3026
6,"hour, TEMP, WSPM -> O3",Aotizhongxin,0.0087,0.2857,1740
7,"hour, TEMP, WSPM -> O3",Changping,0.0086,0.2652,1784
8,"time_slot, NO2 -> O3",Aotizhongxin,0.0480,0.2396,10249
9,"time_slot, NO2 -> O3",Changping,0.0614,0.2260,13351


In [9]:
bootstrap_df = run_bootstrap_validation(
    cleaned_df,
    RESULTS_DIR / 'rfd_bootstrap_summary.csv',
    RESULTS_DIR / 'rfd_bootstrap_iterations.csv',
)
train_test_df = run_train_test_validation(
    cleaned_df,
    RESULTS_DIR / 'rfd_train_test_validation.csv',
)
window_df = run_window_evolution(
    cleaned_df,
    candidate_metrics_df,
    RESULTS_DIR / 'rfd_window_evolution.csv',
    FIGURES_DIR / 'rfd_confidence_over_time.png',
)
violations_summary_df = run_violation_analysis(
    sample_df,
    candidate_metrics_df,
    RESULTS_DIR / 'rfd_violations_summary.csv',
    RESULTS_DIR / 'rfd_top_violating_pairs.csv',
    FIGURES_DIR / 'rfd_violations_by_month_station.png',
)

print('Bootstrap summary')
display(bootstrap_df.round(4))
print('Temporal train-test validation')
display(train_test_df.round(4))
print('Window evolution')
display(window_df.loc[:, [
    'rule_label', 'window_start', 'antecedent_pairs', 'support', 'confidence',
    'violation_rate', 'baseline_confidence', 'baseline_confidence_std',
    'lift', 'abrupt_change'
]].round(4))
print('Violation concentration')
display(violations_summary_df.head(20).round(4))


Bootstrap summary


,rule_label,lhs,rhs,support_mean,support_std,support_ci95_low,support_ci95_high,confidence_mean,confidence_std,confidence_ci95_low,confidence_ci95_high,lift_mean,lift_std,lift_ci95_low,lift_ci95_high,iterations
0,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,0.0250,0.0016,0.0227,0.0282,0.4862,0.0174,0.4580,0.5191,3.3497,0.1420,3.1533,3.5747,30
1,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,0.0180,0.0007,0.0169,0.0193,0.4439,0.0111,0.4272,0.4703,3.0575,0.1054,2.8888,3.2182,30
2,"station, PM2.5 -> PM10","station, PM2.5",PM10,0.0711,0.0030,0.0664,0.0770,0.4359,0.0119,0.4170,0.4628,3.0032,0.0972,2.8468,3.1620,30
3,"station, hour, TEMP, WSPM -> O3","station, hour, TEMP, WSPM",O3,0.0042,0.0001,0.0040,0.0044,0.2836,0.0135,0.2684,0.3091,1.7456,0.0519,1.6672,1.8454,30
4,"station, time_slot, TEMP, WSPM -> O3","station, time_slot, TEMP, WSPM",O3,0.0081,0.0003,0.0076,0.0086,0.2756,0.0133,0.2597,0.3003,1.6942,0.0529,1.6208,1.7924,30
5,"station, time_slot, NO2 -> O3","station, time_slot, NO2",O3,0.0274,0.0006,0.0263,0.0287,0.2362,0.0101,0.2191,0.2558,1.4535,0.0369,1.3696,1.5004,30
6,"station, TEMP, DEWP, WSPM -> PM2.5","station, TEMP, DEWP, WSPM",PM2.5,0.0063,0.0003,0.0058,0.0069,0.1518,0.0066,0.1397,0.1632,1.0720,0.0398,1.0168,1.1632,30


Temporal train-test validation


,rule_label,lhs,rhs,support_train,confidence_train,violation_rate_train,baseline_confidence_train,baseline_confidence_std_train,lift_train,support_test,confidence_test,violation_rate_test,baseline_confidence_test,baseline_confidence_std_test,lift_test,confidence_delta_test_minus_train,lift_delta_test_minus_train
0,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,0.0254,0.4444,0.5556,0.1410,0.0033,3.1510,0.0276,0.5285,0.4715,0.1579,0.0044,3.3464,0.0841,0.1954
1,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,0.0186,0.4100,0.5900,0.1428,0.0028,2.8707,0.0186,0.5004,0.4996,0.1556,0.0028,3.2163,0.0904,0.3457
2,"station, PM2.5 -> PM10","station, PM2.5",PM10,0.0733,0.3993,0.6007,0.1416,0.0028,2.8207,0.0737,0.4846,0.5154,0.1567,0.0028,3.0917,0.0852,0.2710
3,"station, hour, TEMP, WSPM -> O3","station, hour, TEMP, WSPM",O3,0.0041,0.2550,0.7450,0.1581,0.0060,1.6124,0.0045,0.2796,0.7204,0.1499,0.0051,1.8648,0.0246,0.2524
4,"station, time_slot, TEMP, WSPM -> O3","station, time_slot, TEMP, WSPM",O3,0.0080,0.2481,0.7519,0.1565,0.0044,1.5854,0.0088,0.2659,0.7341,0.1505,0.0046,1.7662,0.0178,0.1809
5,"station, time_slot, NO2 -> O3","station, time_slot, NO2",O3,0.0282,0.2223,0.7777,0.1565,0.0026,1.4204,0.0309,0.2187,0.7813,0.1495,0.0023,1.4632,-0.0036,0.0428
6,"station, TEMP, DEWP, WSPM -> PM2.5","station, TEMP, DEWP, WSPM",PM2.5,0.0063,0.1442,0.8558,0.1443,0.0067,0.9994,0.0063,0.1759,0.8241,0.1461,0.0057,1.2041,0.0317,0.2047


Window evolution


/var/folders/hn/yqq1wz1x1qg3gbdk_f7rxb5c0000gn/T/ipykernel_30093/2494017372.py:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(4))


,rule_label,window_start,antecedent_pairs,support,confidence,violation_rate,baseline_confidence,baseline_confidence_std,lift,abrupt_change
0,"station, PM2.5 -> PM10",2013-03-01,53004,0.0494,0.4691,0.5309,0.1046,0.0022,4.4851,False
1,"station, PM2.5 -> PM10",2013-04-01,89360,0.0952,0.3653,0.6347,0.1478,0.0026,2.4712,True
2,"station, PM2.5 -> PM10",2013-05-01,61130,0.0622,0.2328,0.7672,0.1282,0.0018,1.8164,True
3,"station, PM2.5 -> PM10",2013-06-01,47650,0.0527,0.2813,0.7187,0.1247,0.0018,2.2570,False
4,"station, PM2.5 -> PM10",2013-07-01,83135,0.0816,0.3672,0.6328,0.1829,0.0017,2.0078,False
...,...,...,...,...,...,...,...,...,...,...
187,"station, time_slot, PM2.5 -> PM10",2016-10-01,15813,0.0152,0.5170,0.4830,0.1440,0.0040,3.5908,True
188,"station, time_slot, PM2.5 -> PM10",2016-11-01,14134,0.0144,0.5179,0.4821,0.1182,0.0024,4.3826,True
189,"station, time_slot, PM2.5 -> PM10",2016-12-01,15412,0.0144,0.6053,0.3947,0.1120,0.0045,5.4062,True
190,"station, time_slot, PM2.5 -> PM10",2017-01-01,19674,0.0187,0.6123,0.3877,0.1425,0.0049,4.2975,True


Violation concentration


,rule_label,station,month,time_slot,violation_pairs,mean_rhs_abs_diff,max_rhs_abs_diff,share_within_rule
0,"station, PM2.5, NO2 -> PM10",Changping,2014-03,evening,67,167.5672,179.0,0.335
1,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2013-03,afternoon,9,187.0000,209.0,0.045
2,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2013-12,afternoon,7,219.1429,251.0,0.035
3,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2013-05,morning,5,161.8000,166.0,0.025
4,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2013-08,afternoon,5,160.8000,174.0,0.025
5,"station, PM2.5, NO2 -> PM10",Changping,2014-03,morning,4,181.5000,191.0,0.020
6,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2015-04,night,4,179.5000,199.0,0.020
7,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2014-04,evening,4,175.2500,182.0,0.020
8,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2015-04,evening,4,158.7500,163.0,0.020
9,"station, PM2.5, NO2 -> PM10",Aotizhongxin,2015-03,morning,3,216.3333,237.0,0.015


## Step 10: Violation Analysis

In [10]:
selected_violations = select_top_rule_violations(candidate_df, candidate_violations, top_rules=2, per_rule=5)
violations_df = export_violation_examples(
    selected_violations,
    RESULTS_DIR / 'violations_examples.csv',
    limit=10,
)

display(violations_df)


,rule_label,lhs,rhs,row_index_1,row_index_2,datetime_1,datetime_2,station_1,station_2,rhs_value_1,rhs_value_2,rhs_abs_diff,PM2.5_1,PM2.5_2,NO2_1,NO2_2,time_slot_1,time_slot_2
0,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,165,1694,2013-03-04 12:00:00,2013-04-05 19:00:00,Aotizhongxin,Aotizhongxin,28.0,6.0,22.0,13.0,3.0,14.0,12.0,NaN,NaN
1,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,165,1828,2013-03-04 12:00:00,2013-04-08 15:00:00,Aotizhongxin,Aotizhongxin,28.0,149.0,121.0,13.0,15.0,14.0,6.0,NaN,NaN
2,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,165,2539,2013-03-04 12:00:00,2013-04-24 16:00:00,Aotizhongxin,Aotizhongxin,28.0,86.0,58.0,13.0,10.0,14.0,13.0,NaN,NaN
3,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,165,3260,2013-03-04 12:00:00,2013-05-10 17:00:00,Aotizhongxin,Aotizhongxin,28.0,58.0,30.0,13.0,14.0,14.0,20.0,NaN,NaN
4,"station, PM2.5, NO2 -> PM10","station, PM2.5, NO2",PM10,165,8876,2013-03-04 12:00:00,2013-09-10 13:00:00,Aotizhongxin,Aotizhongxin,28.0,12.0,16.0,13.0,9.0,14.0,17.0,NaN,NaN
5,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,165,1828,2013-03-04 12:00:00,2013-04-08 15:00:00,Aotizhongxin,Aotizhongxin,28.0,149.0,121.0,13.0,15.0,NaN,NaN,afternoon,afternoon
6,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,165,2539,2013-03-04 12:00:00,2013-04-24 16:00:00,Aotizhongxin,Aotizhongxin,28.0,86.0,58.0,13.0,10.0,NaN,NaN,afternoon,afternoon
7,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,165,3260,2013-03-04 12:00:00,2013-05-10 17:00:00,Aotizhongxin,Aotizhongxin,28.0,58.0,30.0,13.0,14.0,NaN,NaN,afternoon,afternoon
8,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,165,5189,2013-03-04 12:00:00,2013-06-21 14:00:00,Aotizhongxin,Aotizhongxin,28.0,94.0,66.0,13.0,21.0,NaN,NaN,afternoon,afternoon
9,"station, time_slot, PM2.5 -> PM10","station, time_slot, PM2.5",PM10,165,8876,2013-03-04 12:00:00,2013-09-10 13:00:00,Aotizhongxin,Aotizhongxin,28.0,12.0,16.0,13.0,9.0,NaN,NaN,afternoon,afternoon


## Takeaways

- Candidate RFDs are interpretable but moderately weak on this subset; none behaves like near-deterministic rule.
- Relaxed thresholds increase confidence and support, but still do not yield discovery results above project filter (`0.01`, `0.85`).
- Violating pairs remain useful for UDT framing: they can indicate local inconsistency, unusual pollution dynamics, or missing contextual variables.
- This is approximate consistency analysis, not prediction and not causality.
